# SC4002: Natural Language Processing Project, Logistic Regression (Stage 1)

## Assignment Requirement: Build a system that identifies user's intent and extracts the slots in an English Assistance request. It involves two stages:
1. Task 1: Develop a baseline independently, without Generative AI assistance. (In this Notebook)
2. Task 2: Collaborate with AI to diagnose weaknesses, implement improvements.

Evaluate the resulting models from the above two stages. Submit reproducible notebook evidence, chat history with AI and a report of no more than two pages.

* Data can be found here "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz", en-US.jsonl file


## Step-by-Step procedures to completing Task 1
1. Data Retrieval & Insight
2. Data Cleaning
3. Refining Inputs
4. Building the JointLogisticRegression Model
5. Training and Develop
6. Evaluate test

In [1]:
# Necessary Files needed for the Project
import re
import time
import random
import torch
import scipy.sparse as sp
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# Read json file
import json

# Data Manipulation Libraries
import numpy as np
import pandas as pd
import sklearn
from collections import Counter

# Visualisation Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

In [2]:
# Initialise all global variables
df = train_df = test_df = dev_df = pd.DataFrame()
label_encoder = LabelEncoder()

# Training Variables
number_of_epoch = 12

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python will run the model on:", device)
print("PyTorch version:", torch.__version__)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Python will run the model on: cpu
PyTorch version: 2.15.0


In [3]:
# Step 1: Extracting data from jsonl file to dataframes & insights
try: 
    df = pd.read_json('data/en-US.jsonl', lines=True)
    print(df.head())
    print(df.columns)
except FileNotFoundError:
    print("ERROR: File not found")
except json.JSONDecodeError:
    print("ERROR: Invalid Json format in the file")

   id locale partition scenario             intent  \
0   0  en-US      test    alarm          alarm_set   
1   1  en-US     train    alarm          alarm_set   
2   2  en-US     train    alarm          alarm_set   
3   3  en-US      test    audio  audio_volume_mute   
4   4  en-US     train    audio  audio_volume_mute   

                                   utt  \
0      wake me up at five am this week   
1      wake me up at nine am on friday   
2  set an alarm for two hours from now   
3                                quiet   
4                           olly quiet   

                                           annot_utt  worker_id  
0  wake me up at [time : five am] [date : this week]          1  
1  wake me up at [time : nine am] on [date : friday]          1  
2       set an alarm for [time : two hours from now]          1  
3                                              quiet          1  
4                                         olly quiet          1  
Index(['id', 'locale', 'pa

## Step 2: Data Cleaning

1. Drop columns such as 'locale' and 'worker_id'. These columns are unnecessary to the project.
2. Identify number of 'train', 'test' and 'dev' values in the DataFrame
3. Identify unique 'intent' and 'scenario'
4. Identify 'intent' and 'scenario' counts
5. Insert a new column 'intent_

In [4]:
# Step 1 in Data Cleaning
df = df.drop(columns=['locale', 'worker_id'])
print(df.columns)

Index(['id', 'partition', 'scenario', 'intent', 'utt', 'annot_utt'], dtype='str')


In [5]:
# Step 2 in Data Cleaning
# Print out how many train, test and dev datas in the Dataframe
print(df['partition'].value_counts())

partition
train    11514
test      2974
dev       2033
Name: count, dtype: int64


In [6]:
# Step 3 & 4 in Data Cleaning

# Identifying all values in 'intent' & count
print(df['intent'].unique())
print(df['intent'].value_counts())

<StringArray>
[               'alarm_set',        'audio_volume_mute',
      'iot_hue_lightchange',         'iot_hue_lightoff',
          'iot_hue_lighton',         'iot_hue_lightdim',
             'iot_cleaning',           'calendar_query',
               'play_music',           'general_quirky',
            'general_greet',           'datetime_query',
         'datetime_convert',           'takeaway_query',
             'alarm_remove',              'alarm_query',
               'news_query',           'music_likeness',
              'music_query',          'iot_hue_lightup',
           'takeaway_order',            'weather_query',
           'music_settings',        'audio_volume_down',
             'general_joke',        'music_dislikeness',
       'audio_volume_other',               'iot_coffee',
          'audio_volume_up',              'iot_wemo_on',
             'iot_wemo_off',                 'qa_stock',
               'play_radio',              'social_post',
 'recommendation_

In [7]:
# Step 3 & 4 in Data Cleaning (2)

# Identifying all values in 'scenario' & count
print(df['scenario'].unique())
print(df['scenario'].value_counts())

<StringArray>
[         'alarm',          'audio',            'iot',       'calendar',
           'play',        'general',       'datetime',       'takeaway',
           'news',          'music',        'weather',             'qa',
         'social', 'recommendation',        'cooking',      'transport',
          'email',          'lists']
Length: 18, dtype: str
scenario
calendar          2370
play              2024
qa                1685
email             1381
iot               1107
general            963
weather            855
transport          805
lists              793
news               709
recommendation     596
datetime           578
social             565
alarm              550
music              469
audio              387
takeaway           358
cooking            326
Name: count, dtype: int64


## Step 3: Refining Inputs

1. Separate each partition into their own dataframes (Training, Test, Dev)
2. Encodes each 'intent' into a number (int)
3. Parses each utterance into slot spans
4. Tokenise each word in the utterance
5. Converts each utterance into BIO Tags
6. Build the Vocabulary

In [8]:
# Separating each partition
train_df = df[df['partition'] == 'train'].copy()
test_df = df[df['partition'] == 'test'].copy()
dev_df = df[df['partition'] == 'dev'].copy()

assert len(train_df) == 11514
assert len(test_df) == 2974
assert len(dev_df) == 2033

# Encodes each 'intent' into a number
train_df['intent_index'] = label_encoder.fit_transform(train_df['intent'])
test_df['intent_index'] = label_encoder.transform(test_df['intent'])
dev_df['intent_index'] = label_encoder.transform(dev_df['intent'])

print("Training Set (Top 5)")
for _, row in train_df.head(5).iterrows():
    print(f"Data: {row['intent']} | {row['intent_index']}")
print("\nTest Set (Top 5)")
for _, row in test_df.head(5).iterrows():
    print(f"Data: {row['intent']} | {row['intent_index']}")
print("\nDev Set (Top 5)")
for _, row in dev_df.head(5).iterrows():
    print(f"Data: {row['intent']} | {row['intent_index']}")

Training Set (Top 5)
Data: alarm_set | 2
Data: alarm_set | 2
Data: audio_volume_mute | 4
Data: audio_volume_mute | 4
Data: audio_volume_mute | 4

Test Set (Top 5)
Data: alarm_set | 2
Data: audio_volume_mute | 4
Data: iot_hue_lightchange | 23
Data: iot_hue_lighton | 26
Data: iot_hue_lightoff | 25

Dev Set (Top 5)
Data: iot_hue_lightoff | 25
Data: iot_hue_lightdim | 24
Data: iot_hue_lightdim | 24
Data: iot_cleaning | 21
Data: iot_cleaning | 21


In [9]:
slot_annotation_pattern = re.compile(r"\[([^:\]]+)\s*:\s*([^\]]+)\]")
word_pattern = re.compile(r"\S+")

def parse_annotated_utterance(annotated_utterance):
    original_text_parts = []
    slot_spans = []
    current_character_position = 0
    previous_match_end = 0

    for annotation_match in slot_annotation_pattern.finditer(annotated_utterance):
        text_before_slot = annotated_utterance[previous_match_end:annotation_match.start()]
        original_text_parts.append(text_before_slot)
        current_character_position += len(text_before_slot)

        slot_type = annotation_match.group(1).strip()
        slot_text = annotation_match.group(2)
        slot_start = current_character_position
        slot_end = slot_start + len(slot_text)

        original_text_parts.append(slot_text)
        slot_spans.append((slot_type, slot_start, slot_end))
        current_character_position = slot_end
        previous_match_end = annotation_match.end()

    original_text_parts.append(annotated_utterance[previous_match_end:])
    original_text = "".join(original_text_parts)
    return original_text, slot_spans


def tokenize_with_offsets(utterance):
    token_offsets = []
    for token_match in word_pattern.finditer(utterance):
        token_offsets.append((token_match.group(), token_match.start(), token_match.end()))
    return token_offsets


def convert_spans_to_bio(token_offsets, slot_spans):
    bio_labels = ["O"] * len(token_offsets)

    for slot_type, slot_start, slot_end in slot_spans:
        matching_token_indices = []
        for token_index, (token, token_start, token_end) in enumerate(token_offsets):
            if token_start < slot_end and token_end > slot_start:
                matching_token_indices.append(token_index)

        if not matching_token_indices:
            raise ValueError(f"No token found for slot {slot_type} at {slot_start}:{slot_end}")

        first_token_index = matching_token_indices[0]
        last_token_index = matching_token_indices[-1]
        first_token_start = token_offsets[first_token_index][1]
        last_token_end = token_offsets[last_token_index][2]

        if first_token_start != slot_start or last_token_end != slot_end:
            raise ValueError(f"Slot is not aligned with token boundaries: {slot_type}")

        if any(bio_labels[token_index] != "O" for token_index in matching_token_indices):
            raise ValueError("Overlapping slot annotations are not supported")

        bio_labels[first_token_index] = f"B-{slot_type}"
        for token_index in matching_token_indices[1:]:
            bio_labels[token_index] = f"I-{slot_type}"

    return bio_labels


sample_data = train_df.iloc[0]
sample_text, sample_slot_spans = parse_annotated_utterance(sample_data["annot_utt"])
sample_tokens = tokenize_with_offsets(sample_text)
sample_bio_labels = convert_spans_to_bio(sample_tokens, sample_slot_spans)

print("Original sentence:", sample_text)
print("Tokens:", [token for token, start, end in sample_tokens])
print("BIO labels:", sample_bio_labels)
print("Slot spans:", [(slot_type, sample_text[start:end]) for slot_type, start, end in sample_slot_spans])

Original sentence: wake me up at nine am on friday
Tokens: ['wake', 'me', 'up', 'at', 'nine', 'am', 'on', 'friday']
BIO labels: ['O', 'O', 'O', 'O', 'B-time', 'I-time', 'O', 'B-date']
Slot spans: [('time', 'nine am'), ('date', 'friday')]


In [10]:
COLUMNS = ['id', 'intent_index', 'original_text', 'tokens', 'slots_spans', 'bio_tags']

def build_utt_frame(records, partition_name):
    records = records.to_dict('records')
    rows = []
    for record in records:
        text, slots_spans = parse_annotated_utterance(record['annot_utt'])

        assert text == record['utt'], (f"Text mismatch: {partition_name}, record {record['id']}")
        
        tokens = tokenize_with_offsets(text)
        bio_tags = convert_spans_to_bio(tokens, slots_spans)
        rows.append({
            'id': record['id'],
            'intent_index': record['intent_index'],
            'original_text': text,
            'tokens': tokens,
            'slots_spans': slots_spans,
            'bio_tags': bio_tags}
        )
    print(f"Successfully annoted {partition_name}")
    return pd.DataFrame(rows, columns=COLUMNS)

dataframe_partitions = {
    'train': train_df,
    'test': test_df,
    'dev': dev_df
}

frames = {
    name: build_utt_frame(records, name)
    for name, records in dataframe_partitions.items()
}

training_utt_tokens = frames['train']
test_utt_tokens = frames['test']
dev_utt_tokens = frames['dev']

Successfully annoted train
Successfully annoted test
Successfully annoted dev


In [11]:
PAD, UNK = '<PAD>', '<UNK>'

def token_text(token):
    return token[0] if isinstance (token,tuple) else token

# Build the vocabulary based on training tokens
word_counts = Counter(
    token_text(token)
    for tokens in training_utt_tokens['tokens']
    for token in tokens
)

word2idx = {PAD: 0, UNK: 1}
for word in word_counts: 
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

# Encode tokens as indices (unseen words become <UNK>)
def encode_tokens(tokens):
    return [word2idx.get(token_text(token), word2idx[UNK]) for token in tokens]

for df in (training_utt_tokens, test_utt_tokens, dev_utt_tokens):
    df['tokens_id'] = df['tokens'].apply(encode_tokens)

tag_set = sorted({tag for tags in training_utt_tokens['bio_tags'] for tag in tags})
tag2idx = {tag: idx for idx, tag in enumerate(tag_set)}

def encode_tags(tags):
    return [tag2idx.get(tag, tag2idx.get('O', 0)) for tag in tags]

for df in (training_utt_tokens, test_utt_tokens, dev_utt_tokens):
    df['tag_id'] = df['bio_tags'].apply(encode_tags)

In [12]:
idx2tag = {idx: tag for tag, idx in tag2idx.items()}
NUM_INTENTS = len(label_encoder.classes_)
NUM_TAGS = len(tag2idx)

# Shared features for every token
def token_features(words, i):
    w = words[i]
    return {
        'word.lower': w.lower(),
        'word.istitle': w.istitle(),
        'word.isupper': w.isupper(),
        'word.isdigit': w.isdigit(),
        'word.hasdigit': any(c.isdigit() for c in w),
        'prefix3': w[:3].lower(),
        'suffix3': w[-3:].lower(),
        'pos_from_start': min(i,5),
        'pos_from_end': min(len(words) - 1 - i, 5),
        'prev.lower': words[i-1].lower() if i > 0 else '<BOS>',
        'prev2.lower': words[i-2].lower() if i > 1 else '<BOS>',
        'next.lower': words[i+1].lower() if i < len(words) - 1 else '<EOS>',
        'next2.lower': words[i+2].lower() if i < len(words) - 2 else '<EOS>',
    }

def utterance_token_dicts(tokens):
    words = [token[0] for token in tokens]
    utt_bow = {f"utt:{w.lower()}": 1 for w in words}
    out = []
    for i in range(len(words)):
        feats = token_features(words, i)
        feats.update(utt_bow)
        out.append(feats)
    return out

def collect (utt_df):
    dicts, tag_ids, starts, ends, pos = [], [], [], [], 0
    for tokens, bio in zip(utt_df['tokens'], utt_df['bio_tags']):
        feats = utterance_token_dicts(tokens)
        dicts.extend(feats)
        tag_ids.extend(tag2idx.get(t, tag2idx['O']) for t in bio)
        starts.append(pos)
        pos += len(feats)
        ends.append(pos)
    return dicts, np.array(tag_ids), np.array(starts), np.array(ends)

def bio_to_spans(tags):
    spans, current = [], None
    for i, tag in enumerate(tags):
        if tag == 'O':
            if current:
                spans.append(current)
            current = None
            continue
        prefix, slot = tag.split('-', 1)
        if prefix == 'B' or current is None or current[0] != slot:
            if current:
                spans.append(current)
            current = (slot, i, i)
        else:
            current = (slot, current[1], i)
    if current:
        spans.append(current)
    return spans

def span_prf(gold_seqs, pred_seqs):
    tp = fp = fn = 0
    for gold_tags, pred_tags in zip(gold_seqs, pred_seqs):
        gold = set(bio_to_spans(gold_tags))
        pred = set(bio_to_spans(pred_tags))
        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

parts = {'train': training_utt_tokens, 'dev': dev_utt_tokens, 'test':test_utt_tokens}
collected = {name: collect(utt_df) for name, utt_df in parts.items()}
feat_vec = DictVectorizer().fit(collected['train'][0]) # fit on training only

splits = {}
for name, utt_df in parts.items():
    dicts, tags, starts, ends = collected[name]
    splits[name] = {
        'utt_df': utt_df,
        'X': feat_vec.transform(dicts).tocsr(),
        'tags': tags,
        'starts': starts,
        'ends': ends,
        'intents':utt_df['intent_index'].astype(int).values,
    }

N_FEATURES = splits['train']['X'].shape[1]
print(f"Number of features: {N_FEATURES}")

Number of features: 31766


In [13]:
# Batching: Gathering Token rows and build the utterance pooling matrix
def to_torch_sparse(m):
    m = m.tocoo()
    idx = torch.tensor(np.vstack([m.row, m.col]), dtype=torch.long)
    return torch.sparse_coo_tensor(idx, torch.tensor(m.data, dtype=torch.float32), m.shape).coalesce()

def make_collate(split):
    X, tags, starts, ends, intents = (split['X'], split['tags'], split['starts'], split['ends'], split['intents'])

    def collate(utt_ids):
        utt_ids = np.array(utt_ids)
        lengths = ends[utt_ids] - starts[utt_ids]
        tok_idx = np.concatenate([np.arange(starts[u], ends[u]) for u in utt_ids])
        B, T = len(utt_ids), len(tok_idx)
        utt_of_token = np.repeat(np.arange(B), lengths)
        P = sp.csr_matrix((1.0/lengths[utt_of_token], (utt_of_token, np.arange(T))),shape=(B,T))
        return {
            'X': to_torch_sparse(X[tok_idx]),
            'P': to_torch_sparse(P),
            'tags': torch.tensor(tags[tok_idx], dtype=torch.long),
            'intents': torch.tensor(intents[utt_ids], dtype=torch.long),
            'lengths': lengths,
        }
    return collate

class IndexDataset(Dataset):
    def __init__(self, n):
        self.n = n
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        return i

loaders = {
    'train': DataLoader(IndexDataset(len(splits['train']['intents'])), batch_size = 64, shuffle=True, collate_fn=make_collate(splits['train'])),
    'dev': DataLoader(IndexDataset(len(splits['dev']['intents'])), batch_size = 256, shuffle=False, collate_fn=make_collate(splits['dev'])),
    'test': DataLoader(IndexDataset(len(splits['test']['intents'])), batch_size = 256, shuffle=False, collate_fn=make_collate(splits['test']))
}

## Step 4: Building the JointLogisticRegression model

In [14]:
# Part 6 of Machine Learning

# Joint Model Framework
class JointLogisticRegression(nn.Module):
    def __init__(self, n_features, n_tags, n_intents):
        super().__init__()
        self.n_tags = n_tags
        self.weight = nn.Parameter(torch.randn(n_features, n_tags + n_intents) * 0.01)
        self.bias = nn.Parameter(torch.zeros(n_tags + n_intents))

    def forward(self, X, P):
        Z = torch.sparse.mm(X, self.weight) + self.bias # (T, n_tags + n_intents)
        slot_logits = Z[:, :self.n_tags] # (T, n_tags)
        intent_logits = torch.sparse.mm(P, Z[:, self.n_tags:]) # (B, n_intents)
        return intent_logits, slot_logits

model = JointLogisticRegression(N_FEATURES, NUM_TAGS, NUM_INTENTS).to(device)
optimizer = Adam(model.parameters(), lr=1e-2)
ce_intent = nn.CrossEntropyLoss()
ce_slot = nn.CrossEntropyLoss()

# Epoch Loops
def run_train_epoch():
    model.train()
    total, batches = 0.0, 0
    for batch in loaders['train']:
        intent_logits, slot_logits = model(batch['X'].to(device), batch['P'].to(device))
        loss = (ce_intent(intent_logits, batch['intents'].to(device)) + ce_slot(slot_logits, batch['tags'].to(device)))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
        batches += 1
    return total / batches

@torch.no_grad()
def run_eval_loss(loader):
    model.eval()
    total, batches = 0.0, 0
    for batch in loader:
        intent_logits, slot_logits = model(batch['X'].to(device), batch['P'].to(device))
        loss = (ce_intent(intent_logits, batch['intents'].to(device)) + ce_slot(slot_logits, batch['tags'].to(device)))
        total += loss.item()
        batches += 1
    return total / batches

@torch.no_grad()
def predict_split(name):
    model.eval()
    intent_preds, slot_preds = [], []
    for batch in loaders[name]:
        intent_logits, slot_logits = model(batch['X'].to(device), batch['P'].to(device))
        intent_preds.extend(intent_logits.argmax(1).tolist())
        tag_ids = slot_logits.argmax(1).tolist()
        start = 0

        for n in batch['lengths']:
            slot_preds.append([idx2tag[t] for t in tag_ids[start:start + n]])
            start += n
    return intent_preds, slot_preds

def evaluate(name):
    intent_preds, slot_preds = predict_split(name)
    intent_acc = accuracy_score(splits[name]['intents'], intent_preds)
    p, r, f = span_prf(splits[name]['utt_df']['bio_tags'], slot_preds)
    return intent_acc, p, r, f

## Step 5: Epoch Training & Model Development

In [15]:
# Epoch Loop with Timing
history = {'epoch': [], 'train_loss': [], 'dev_loss': [], 'intent_acc': [], 'combined': []}

best_combined = float('-inf')
best_epoch = None
total_train_time = 0.0
total_eval_time = 0.0

for epoch in range(1, number_of_epoch + 1):
    t0 = time.perf_counter()
    train_loss = run_train_epoch()
    train_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    dev_loss = run_eval_loss(loaders['dev'])
    intent_acc, p, r, f = evaluate('dev')
    eval_time = time.perf_counter() - t0

    total_train_time += train_time
    total_eval_time += eval_time

    epoch_time = train_time + eval_time
    combined = (intent_acc * 100 + f * 100) / 2

    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['dev_loss'].append(dev_loss)
    history['intent_acc'].append(intent_acc * 100)
    history['combined'].append(combined)
        
    is_best = combined > best_combined
    if is_best:
        best_combined = combined
        best_epoch = epoch
        torch.save(model.state_dict(), 'joint_logreg.pt')
    print(f"Epoch {epoch:02d} | train loss: {train_loss:.4f} | dev loss: {dev_loss:.4f} | "
          f"intent: {intent_acc * 100:.2f}% | slot F1: {f * 100:.2f} | "
          f"combined: {combined:.2f} | time: {epoch_time:.1f}s") 
print(f"Saved Best epoch into 'joint_logreg.pt'")
print(f"Best epoch: {best_epoch} (dev combined {best_combined:.2f}%)")
print(f"Total Training Time: {total_train_time:.2f}s | Total Dev Eval Time: {total_eval_time:.2f}s")

/var/folders/6d/y_vj97ds0xn190m3ychbc82m0000gn/T/ipykernel_93288/851763543.py:5: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:923.)
  return torch.sparse_coo_tensor(idx, torch.tensor(m.data, dtype=torch.float32), m.shape).coalesce()


Epoch 01 | train loss: 2.5735 | dev loss: 1.2731 | intent: 81.26% | slot F1: 67.76 | combined: 74.51 | time: 1.7s
Epoch 02 | train loss: 0.7142 | dev loss: 0.9854 | intent: 83.92% | slot F1: 73.02 | combined: 78.47 | time: 1.7s
Epoch 03 | train loss: 0.4018 | dev loss: 0.8935 | intent: 84.60% | slot F1: 74.96 | combined: 79.78 | time: 1.7s
Epoch 04 | train loss: 0.2675 | dev loss: 0.8717 | intent: 84.55% | slot F1: 75.54 | combined: 80.05 | time: 1.7s
Epoch 05 | train loss: 0.1946 | dev loss: 0.8601 | intent: 85.24% | slot F1: 75.68 | combined: 80.46 | time: 1.7s
Epoch 06 | train loss: 0.1501 | dev loss: 0.8626 | intent: 84.85% | slot F1: 75.24 | combined: 80.04 | time: 1.7s
Epoch 07 | train loss: 0.1207 | dev loss: 0.8681 | intent: 84.55% | slot F1: 75.47 | combined: 80.01 | time: 1.6s
Epoch 08 | train loss: 0.0998 | dev loss: 0.8729 | intent: 84.46% | slot F1: 75.67 | combined: 80.06 | time: 1.6s
Epoch 09 | train loss: 0.0845 | dev loss: 0.8728 | intent: 85.29% | slot F1: 75.82 | com

## Step 6: Evaluate Test

In [16]:
# Test with The best checkpoint
model.load_state_dict(torch.load('joint_logreg.pt', map_location=device))
print(f"Loaded best epoch {best_epoch}")

t0 = time.perf_counter()
intent_acc, p, r, f = evaluate('test')
test_time = time.perf_counter() - t0

print(f"\nTest (best epoch {best_epoch}), intent acc: {intent_acc * 100:.2f}% | slot F1: {f * 100:.2f}% | "
      f"Test eval time: {test_time:.2f}s")

Loaded best epoch 9

Test (best epoch 9), intent acc: 83.83% | slot F1: 71.07% | Test eval time: 0.05s


In [17]:
n_params = sum (p.numel() for p in model.parameters())
n_weight = model.weight.numel()
n_bias = model.bias.numel()

print(f"Features: {N_FEATURES} | Tags: {NUM_TAGS} | Intents: {NUM_INTENTS}")
print(f"Weight Parameters: {n_weight} | Bias Parameters: {n_bias}")
print(f"Total traininable parameters: {n_params:,}")

Features: 31766 | Tags: 107 | Intents: 60
Weight Parameters: 5304922 | Bias Parameters: 167
Total traininable parameters: 5,305,089
